## 3.15 Random Values

**Homework: Popcorn Hack 4, SFI Backend QA Simulator**

**Task from the lesson:** build a randomized QA simulator for the SFI backend. It uses the part records, the backend actions, the list of existing spec numbers, and repeated test runs. It has to handle a duplicate spec on create, and print which record and action were picked and what happened.

**How I solved it:** `random.choice` picks one part and one action for each test run. A procedure named `run_action` decides the result from the action and from whether that spec number is already in the backend. Create and delete change the backend list, so later runs see the change. That is what makes it a simulator and not just random printing.

| Action | Spec is in the backend | Spec is not in the backend |
| --- | --- | --- |
| `GET/search` | 200 OK, record found | 404 Not Found |
| `POST/create` | 409 Conflict, duplicate spec | 201 Created, spec added |
| `PUT/update` | 200 OK, record updated | 404 Not Found |
| `DELETE/remove` | 204 No Content, spec removed | 404 Not Found |

In [ ]:
# CODE_RUNNER: Run it a few times and watch the results change. Then set test_count to 10.
import random

parts = [
    {
        "product_name": "Replacement Flywheels",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    }
]

actions = [
    "GET/search",
    "POST/create",
    "PUT/update",
    "DELETE/remove"
]

existing_spec_numbers = ["1.1", "2.1"]
test_count = 5

def run_action(part, action):
    spec = part["spec_number"]
    in_backend = spec in existing_spec_numbers

    if action == "GET/search":
        if in_backend:
            return "200 OK, record found"
        return "404 Not Found, no record with spec " + spec

    if action == "POST/create":
        if in_backend:
            return "409 Conflict, spec " + spec + " already exists"
        existing_spec_numbers.append(spec)
        return "201 Created, spec " + spec + " added"

    if action == "PUT/update":
        if in_backend:
            return "200 OK, record updated"
        return "404 Not Found, nothing to update"

    if action == "DELETE/remove":
        if in_backend:
            existing_spec_numbers.remove(spec)
            return "204 No Content, spec " + spec + " removed"
        return "404 Not Found, nothing to delete"

    return "400 Bad Request, unknown action"

print("SFI Backend QA Simulator")
print("Specs in the backend at the start: " + str(existing_spec_numbers))
print("")

passed = 0
for test_number in range(1, test_count + 1):
    part = random.choice(parts)
    action = random.choice(actions)
    result = run_action(part, action)
    if result.startswith("2"):
        passed = passed + 1
    print("Test " + str(test_number) + ":")
    print("  Part:   " + part["product_name"] + " (spec " + part["spec_number"] + ")")
    print("  Action: " + action)
    print("  Result: " + result)

print("")
print("Specs in the backend at the end: " + str(existing_spec_numbers))
print("Successful calls: " + str(passed) + " out of " + str(test_count))

### Output and What It Shows

Every run is different, which is the point of using `random`. One sample run gave this:

~~~text
Test 1: Racing Flywheel Record (spec 2.1), POST/create  -> 409 Conflict, spec 2.1 already exists
Test 2: Racing Flywheel Record (spec 2.1), GET/search   -> 200 OK, record found
Test 3: Racing Flywheel Record (spec 2.1), GET/search   -> 200 OK, record found
Test 4: Multiple Disc Clutch Assemblies (spec 1.2), GET/search -> 404 Not Found
Test 5: Multiple Disc Clutch Assemblies (spec 1.2), GET/search -> 404 Not Found
Successful calls: 2 out of 5
~~~

Things to notice:

* Spec 1.2 starts outside the backend, so searching for it gives 404 until a create adds it.
* Creating a spec that is already there gives 409, which is the duplicate handling the task asked for.
* The list of specs printed at the end is usually different from the one printed at the start, because create and delete really change it.

### Submission

* Commit and push the portfolio or blog page.
* Open the published page in your browser and confirm it loads.
* Copy the full published URL, not the GitHub editor or repository URL.
* Paste that URL into the built-in Link Submission form directly below the lesson.
* In the description, write: `3.15 Random SFI backend hacks 1-4 complete`.